# 01 — Forward verification and recoverability

**Outputs:** analytical attenuation, independent linear grid convergence, nonlinear space/time convergence, a frozen numerical reference, and an example independent continuation of an archived reconstructed initial field.

Run this before generating nonlinear observations. A small terminal network error is different from a small independently propagated error. The latter starts from the recovered initial trace and solves the forward problem without the network.

**Resume revision:** your 401/801/1601-node reference has already passed the declared `1e-4` sensor-norm criterion. This notebook loads and verifies that exact reference. It does not overwrite the data used by your completed nonlinear trainings. The other inexpensive diagnostics may be repeated independently.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "inverse_study").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Extract the entire study ZIP and open this notebook from its folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from inverse_study.common import *
from inverse_study import numerics as num
from inverse_study import workflows as flow
RESULTS.mkdir(exist_ok=True)
plt.rcParams.update({"font.size": 11, "figure.dpi": 110, "axes.grid": True,
                     "grid.alpha": 0.2, "savefig.bbox": "tight"})
print("Study folder:", ROOT)
from inverse_study import completion as study
from inverse_study import completion_checks as checks
REDUCED = RESULTS / "reduced_plan"
REDUCED.mkdir(parents=True, exist_ok=True)
study.verify_protocol_lock(require_reference=True)


In [ ]:
software_checks = num.run_self_checks()
display(pd.Series(software_checks).to_frame("software_check"))
(RESULTS / "verification").mkdir(exist_ok=True)
write_json(REDUCED / "numerical_self_checks.json", software_checks)

## 1. Known modal attenuation and measurement sensitivity

The original smooth case has modes $(1,1)$ and $(2,3)$, amplitudes 1 and 0.25, and exposure $\mathrm{Fo}T=0.02$. A mode is attenuated by $\exp[-\lambda_k\mathrm{Fo}T]$. The two-mode calculation below is a **known-family diagnostic**, not the primary inverse competitor. The primary baseline in 02 admits a rich product-sine basis.

In [ ]:
p = problem_spec("smooth2d")
modes = np.asarray(p["modes"])
lam = num.laplacian_eigenvalues(modes)
attenuation = np.exp(-p["Fo"] * p["T"] * lam)
table = pd.DataFrame({"mode": [str(tuple(map(int,k))) for k in modes], "lambda": lam,
                      "initial_amplitude": p["amplitudes"], "attenuation": attenuation,
                      "terminal_amplitude": np.asarray(p["amplitudes"]) * attenuation,
                      "inverse_amplification": 1 / attenuation})
display(table)
table.to_csv(REDUCED / "modal_attenuation.csv", index=False)
exposure = np.linspace(0, .1, 250)
fig, ax = plt.subplots(figsize=(7, 4))
for j, mode in enumerate(modes):
    ax.semilogy(exposure, p["amplitudes"][j] * np.exp(-lam[j] * exposure),
                label=f"Mode {tuple(map(int,mode))}", linestyle=["-", "--"][j])
ax.axvline(.02, color="k", lw=1, label="Original exposure")
ax.set(xlabel="Diffusion exposure Fo*T", ylabel="Terminal modal amplitude")
ax.legend(); plt.show()
fig.savefig(REDUCED / "modal_attenuation.svg")

In [ ]:
rank_rows = []
for n in [5, 9, 17, 33]:
    _, X = tensor_grid(2, n, interior=True)
    for kmax in [4, 8, 12]:
        A = num.forward_matrix(p, X, num.full_sine_modes(2, kmax))
        s = np.linalg.svd(A, compute_uv=False)
        rank_rows.append({"full_grid_n": n, "n_interior": len(X), "basis_modes": kmax**2,
                          "numerical_rank": int(np.sum(s > s[0] * max(A.shape) * np.finfo(float).eps)),
                          "smallest_over_largest_singular_value": float(s[-1]/s[0])})
rank_table = pd.DataFrame(rank_rows)
display(rank_table)
rank_table.to_csv(REDUCED / "finite_sensor_rank.csv", index=False)
print("Finite rank does not identify an unrestricted infinite-dimensional initial field.")

## 2. Independent linear forward grid convergence

The verification operator is a centered finite-difference Laplacian propagated by a matrix exponential. It is independent of the PINN and the direct analytical modal evaluation. Compare progressively refined grids against the continuous exact solution; do not use the same discrete operator on both sides of the check.

In [ ]:
linear_rows = []
for n in [17, 33, 65]:
    axes, X = tensor_grid(2, n)
    initial = exact_linear(p, X).reshape(n, n)
    solution = num.forward_linear_grid(initial, axes, p["Fo"], p["T"])
    linear_rows.append({"nodes_per_axis": n,
                        "terminal_relative_l2": relative_l2(solution["terminal"].ravel(), exact_linear(p, X, p["T"])),
                        "wall_seconds": solution["wall_seconds"]})
linear_table = pd.DataFrame(linear_rows)
display(linear_table)
assert np.all(np.diff(linear_table.terminal_relative_l2) < 0)
linear_table.to_csv(REDUCED / "linear_forward_convergence.csv", index=False)

## 3. Reuse the verified nonlinear reference

The supplied archive already contains spatial meshes 401/801/1601, time-step caps 0.01/0.005/0.0025, BDF tolerances `rtol=1e-9`, `atol=1e-11`, and a passed `1e-4` relative sensor-difference gate. The last spatial sensor difference is approximately 0.0012721; the recorded empirical numerical floor is approximately 0.0025441.

These are empirical discretization checks, not a rigorous error bound. The code verifies the frozen reference hash and solver identity. A future refinement study must use a separate output location and must not silently replace the reference underlying this cohort.


In [ ]:
study.verify_protocol_lock(require_reference=True)
reference, reference_gate = flow.load_reference(require_pass=True)
reference_table = pd.read_csv(RESULTS / "verification/nonlinear_reference_convergence.csv")
display(reference_table)
display(pd.Series(reference_gate).to_frame("value"))
print("Reusing the existing verified reference; no forward-reference sweep was launched.")
fig, ax = plt.subplots(figsize=(7, 4))
for t, profile in zip(reference["t"], reference["u"]):
    ax.plot(reference["x"], profile, label=f"t={t:g}")
ax.set(xlabel="x", ylabel="Temperature", title="Preserved independent nonlinear reference")
ax.legend(ncol=3); plt.show()


## 4. Independent replay of an archived reconstruction

This recomputes a diagnostic from an existing initial field. It does not retrain that model or validate its observation protocol. The boundary mismatch is reported before imposing the known exact boundary for the independent forward solution. Repeat replay at more than one grid before treating the number as final.

In [ ]:
legacy = DATA / "legacy_runs/noise_sweep__smooth2d__d2__obs33__noise000p0"
with np.load(legacy / "predictions.npz", allow_pickle=False) as z:
    oldX = z["X_space"]; old0 = z["theta0_pred"].ravel(); oldT = z["thetaT_pred"].ravel()
old_axes = tuple(np.unique(oldX[:, j]) for j in range(2))
old_field = old0.reshape(tuple(map(len, old_axes)))
from scipy.interpolate import RegularGridInterpolator
interpolator = RegularGridInterpolator(old_axes, old_field)
closure_rows = []
for n in [33, 65, 101]:
    axes, X = tensor_grid(2, n)
    replay = num.forward_linear_grid(interpolator(X).reshape(n, n), axes, p["Fo"], p["T"])
    closure_rows.append({"replay_nodes": n,
                         "independent_terminal_relative_l2": relative_l2(replay["terminal"].ravel(), exact_linear(p, X, p["T"])),
                         "initial_boundary_linf": replay["initial_boundary_linf"],
                         "status": "legacy_postprocessing_diagnostic"})
display(pd.DataFrame(closure_rows))
pd.DataFrame(closure_rows).to_csv(REDUCED / "legacy_clean2d_forward_replay.csv", index=False)
print("Original network-terminal relative L2:", relative_l2(oldT, exact_linear(p, oldX, p["T"])))